In [0]:
%run '../includes/configuration'

In [0]:
%run '../includes/common_functions'

### Ingestión del archivo movie_cast.json

In [0]:
dbutils.widgets.text("p_environment","")
v_environment = dbutils.widgets.get("p_environment")

In [0]:
dbutils.widgets.text("p_file_date","2024-12-23")
v_file_date = dbutils.widgets.get("p_file_date")

##### Paso 1. Leer el archivo

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

In [0]:
movies_casts_schema = StructType(fields=[
    StructField("movieId", IntegerType(), True),
    StructField("personId", IntegerType(), True),
    StructField("characterName", StringType(), True),
    StructField("genderId", IntegerType(), True),
    StructField("castOrder", IntegerType(), True)
])

In [0]:
movies_casts_df = spark.read \
                .schema(movies_casts_schema) \
                .option("multiline",True) \
                .json(f"{bronze_folder_path}/{v_file_date}/movie_cast.json")

##### Paso 2 - Renombrar las columnas y añadir nuevas columnas

In [0]:
from pyspark.sql.functions import col, concat, current_timestamp, lit, trim

In [0]:
movies_casts_with_columns_df = movies_casts_df \
                        .withColumnRenamed("movieId","movie_id") \
                        .withColumnRenamed("personId","person_id") \
                        .withColumnRenamed("characterName","character_name") \
                        .withColumn("ingestion_date", current_timestamp()) \
                        .withColumn("environment", lit(v_environment)) \
                        .withColumn("file_date", lit(v_file_date))

#####Paso 3 - Eliminar columnas a no usar

In [0]:
from pyspark.sql.functions import col

In [0]:
movies_casts_final_df = movies_casts_with_columns_df.drop(col("genderId"),col("castOrder"))

#####Paso 4 - Escribir la salida en formato parquet

In [0]:
#overwrite_partition("movie_silver","movies_casts","file_date",v_file_date)

In [0]:
merge_condition = "tgt.movie_id = src.movie_id AND tgt.person_id = src.person_id AND tgt.file_date = src.file_date"
merge_delta_lake(movies_casts_final_df,"movie_silver","movies_casts",merge_condition,"file_date")

In [0]:
%sql
SELECT file_date, COUNT(1) FROM movie_silver.movies_casts
GROUP BY file_date

file_date,COUNT(1)
2024-12-30,5000
2024-12-23,15000
2024-12-16,30000
